<a href="https://colab.research.google.com/github/muqaddaszaheer/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muqaddaszaheer/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use Logistic Regression because it is a simple and interpretable classification method. It fits this baseline task because the goal is to classify content into a review-priority outcome using observable signals such as staleness and CTR.

The model is used for directional decision support and comparison with the Week-4 baseline. I will not treat the model output as a prediction of future performance.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

repo_url = "https://github.com/muqaddaszaheer/flyrank-ml-internship.git"
repo_dir = "/content/flyrank-ml-internship"

if not os.path.exists(repo_dir):
    !git clone {repo_url} {repo_dir}

os.chdir(repo_dir)

print("Repository:", os.getcwd())
print(
    "Dataset exists:",
    os.path.exists("data/raw/content_refresh_anonymized.csv")
)


Repository: /content/flyrank-ml-internship
Dataset exists: True


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a stratified train/test split so that the classes are represented in both sets. The split is fixed with a random state for reproducibility.

The model and baseline will be evaluated on the same test data and with the same metric. This makes the comparison more consistent and avoids comparing results from different samples.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 — Section 2: Split design and data preparation

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

# ---------------------------------------------------------
# Find the repository
# ---------------------------------------------------------

repo_dir = "/content/flyrank-ml-internship"

if not os.path.exists(repo_dir):
    raise FileNotFoundError(
        "Repository not found at /content/flyrank-ml-internship"
    )

os.chdir(repo_dir)

# ---------------------------------------------------------
# Find the dataset
# ---------------------------------------------------------

data_path = "data/raw/content_refresh_anonymized.csv"

if not os.path.exists(data_path):
    raise FileNotFoundError(
        "Dataset was not found inside the repository at: "
        + data_path
    )

print("Repository:", os.getcwd())
print("Dataset:", data_path)

# ---------------------------------------------------------
# Load dataset
# ---------------------------------------------------------

df = pd.read_csv(data_path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

# ---------------------------------------------------------
# Check required columns
# ---------------------------------------------------------

required_columns = [
    "days_since_last_update",
    "ctr"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# ---------------------------------------------------------
# Convert signals to numeric
# ---------------------------------------------------------

df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"],
    errors="coerce"
)

df["ctr"] = pd.to_numeric(
    df["ctr"],
    errors="coerce"
)

# ---------------------------------------------------------
# Keep usable rows
# ---------------------------------------------------------

model_data = df[
    ["days_since_last_update", "ctr"]
].dropna().copy()

print("Rows with both signals:", len(model_data))

# ---------------------------------------------------------
# Recreate Week-4 baseline thresholds
# ---------------------------------------------------------

stale_threshold = model_data[
    "days_since_last_update"
].quantile(0.75)

low_ctr_threshold = model_data[
    "ctr"
].quantile(0.25)

# ---------------------------------------------------------
# Create baseline signals
# ---------------------------------------------------------

model_data["stale_signal"] = (
    model_data["days_since_last_update"]
    >= stale_threshold
).astype(int)

model_data["low_ctr_signal"] = (
    model_data["ctr"]
    <= low_ctr_threshold
).astype(int)

model_data["baseline_score"] = (
    model_data["stale_signal"]
    + model_data["low_ctr_signal"]
)

# Review if at least one signal is present
model_data["baseline_target"] = (
    model_data["baseline_score"] >= 1
).astype(int)

# ---------------------------------------------------------
# Features and target
# ---------------------------------------------------------

X = model_data[
    ["days_since_last_update", "ctr"]
]

y = model_data["baseline_target"]

# ---------------------------------------------------------
# Train/test split
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

print("\nSplit completed successfully.")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

print("\nTest target distribution:")
print(y_test.value_counts().sort_index())

print("\nStale threshold:", round(stale_threshold, 2))
print("Low CTR threshold:", round(low_ctr_threshold, 4))

print("\nSection 2 completed successfully.")


Repository: /content/flyrank-ml-internship
Dataset: data/raw/content_refresh_anonymized.csv
Rows: 30000
Columns: 44
Rows with both signals: 30000

Split completed successfully.
Training rows: 24000
Test rows: 6000

Training target distribution:
baseline_target
0     8554
1    15446
Name: count, dtype: int64

Test target distribution:
baseline_target
0    2139
1    3861
Name: count, dtype: int64

Stale threshold: 104.0
Low CTR threshold: 0.0

Section 2 completed successfully.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I compare Logistic Regression with the Week-4 baseline on the same held-out test set. The main comparison uses accuracy, precision, recall, and F1 so the result is not judged using only one number.

The baseline is a simple rule based on the two observed signals, while Logistic Regression learns a linear decision boundary from the same signals.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 — Section 3: Train model and compare with baseline

# Train Logistic Regression
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

# Model predictions
model_pred = model.predict(X_test)

# Baseline predictions on exactly the same test rows
baseline_pred = (
    (
        (X_test["days_since_last_update"] >= stale_threshold)
        |
        (X_test["ctr"] <= low_ctr_threshold)
    )
).astype(int)

# Calculate metrics
model_accuracy = accuracy_score(y_test, model_pred)
model_precision = precision_score(
    y_test,
    model_pred,
    zero_division=0
)
model_recall = recall_score(
    y_test,
    model_pred,
    zero_division=0
)
model_f1 = f1_score(
    y_test,
    model_pred,
    zero_division=0
)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)
baseline_precision = precision_score(
    y_test,
    baseline_pred,
    zero_division=0
)
baseline_recall = recall_score(
    y_test,
    baseline_pred,
    zero_division=0
)
baseline_f1 = f1_score(
    y_test,
    baseline_pred,
    zero_division=0
)

comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "accuracy": [
        baseline_accuracy,
        model_accuracy
    ],
    "precision": [
        baseline_precision,
        model_precision
    ],
    "recall": [
        baseline_recall,
        model_recall
    ],
    "f1": [
        baseline_f1,
        model_f1
    ]
})

print("Model vs Baseline")
print("=" * 70)
print(
    comparison.round(4).to_string(index=False)
)

print("\nLogistic Regression coefficients:")
print(
    pd.DataFrame({
        "feature": X.columns,
        "coefficient": model.coef_[0]
    }).to_string(index=False)
)

print("\nSection 3 completed successfully.")


Model vs Baseline
             method  accuracy  precision  recall     f1
    Week-4 baseline    1.0000     1.0000   1.000 1.0000
Logistic Regression    0.8052     0.8202   0.893 0.8551

Logistic Regression coefficients:
               feature  coefficient
days_since_last_update     0.068539
                   ctr    -2.809776

Section 3 completed successfully.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The error analysis focuses on the cases where the model disagrees with the observed target. These cases show where a simple linear model does not reproduce the baseline rule exactly.

The model uses the two available signals, so its interpretation is limited to staleness and CTR. A disagreement does not automatically mean that the model or baseline is wrong; it shows where their decision rules differ.

The results are directional and should be reviewed before any content action is taken.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 — Section 4: Error analysis and interpretation

# Build an error-analysis table
error_analysis = X_test.copy()

error_analysis["actual"] = y_test.values
error_analysis["model_prediction"] = model_pred
error_analysis["baseline_prediction"] = baseline_pred

# Identify model errors
model_errors = error_analysis[
    error_analysis["actual"] != error_analysis["model_prediction"]
].copy()

print("Error analysis")
print("=" * 70)
print("Test rows:", len(error_analysis))
print("Model errors:", len(model_errors))
print(
    "Model error rate:",
    round(len(model_errors) / len(error_analysis), 4)
)

if len(model_errors) > 0:
    print("\nFirst 10 model errors:")
    print(
        model_errors.head(10).to_string(index=False)
    )
else:
    print("\nNo model errors on the test set.")

# Compare model and baseline predictions
different_predictions = (
    error_analysis["model_prediction"]
    != error_analysis["baseline_prediction"]
).sum()

print(
    "\nRows where model and baseline disagree:",
    different_predictions
)

# Interpret model coefficients
coefficient_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
})

print("\nFeature interpretation:")
print(
    coefficient_table.to_string(index=False)
)

print("\nInterpretation:")
for _, row in coefficient_table.iterrows():
    direction = (
        "increases"
        if row["coefficient"] > 0
        else "decreases"
    )

    print(
        f"- {row['feature']} {direction} the model's "
        f"tendency toward the review class."
    )

print(
    "\nThe model should be treated as directional decision support, "
    "not as proof that a content item needs a specific intervention."
)

print("\nSection 4 completed successfully.")


Error analysis
Test rows: 6000
Model errors: 1169
Model error rate: 0.1948

First 10 model errors:
 days_since_last_update  ctr  actual  model_prediction  baseline_prediction
                      6 0.00       1                 0                    1
                     20 0.07       0                 1                    0
                      6 0.00       1                 0                    1
                    104 2.32       1                 0                    1
                      8 0.00       1                 0                    1
                      8 0.00       1                 0                    1
                     22 0.16       0                 1                    0
                     20 0.14       0                 1                    0
                     20 0.04       0                 1                    0
                     22 0.19       0                 1                    0

Rows where model and baseline disagree: 1169

Feature interpreta

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.